# ETAPA 5 - Fine-tuning QLoRA do Qwen3-8B
Treinamento acadêmico real em dados sintéticos do Hospital TechCare. Use Google Colab com **Tesla T4 ou GPU compatível com pelo menos 14 GB de VRAM** e a versão de runtime **2026.07 (Python 3.12)**.

In [ ]:
!nvidia-smi

In [ ]:
import platform
print(platform.python_version())
assert platform.python_version_tuple()[:2] == ('3', '12'), 'Selecione o runtime 2026.07 com Python 3.12'

In [ ]:
import os, subprocess
REPOSITORY = 'https://github.com/mo1sess/tech-challenge-fase3.git'
PROJECT = '/content/tech-challenge-fase3'
if not os.path.exists(PROJECT):
    subprocess.run(['git', 'clone', REPOSITORY, PROJECT], check=True)
else:
    subprocess.run(['git', '-C', PROJECT, 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
print(os.getcwd())
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)

In [ ]:
%pip install -q -e . -r requirements/gpu-colab-kaggle.txt

In [ ]:
import torch, transformers, peft, trl, accelerate, bitsandbytes, datasets
print({
    'torch': torch.__version__, 'transformers': transformers.__version__,
    'peft': peft.__version__, 'trl': trl.__version__,
    'accelerate': accelerate.__version__, 'bitsandbytes': bitsandbytes.__version__,
    'datasets': datasets.__version__, 'cuda': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})
assert torch.cuda.is_available(), 'GPU CUDA obrigatória'

In [ ]:
!python scripts/prepare_finetuning_data.py

In [ ]:
!python scripts/validate_finetuning_data.py

## Treinamento oficial
A próxima célula executa QLoRA real, valida ao final de cada época e salva somente o adapter separado. Não interrompa o runtime.

In [ ]:
!python scripts/train_qlora.py

In [ ]:
import glob, json, pathlib
manifests = sorted(glob.glob('outputs/training/training-*/run_manifest.json'))
complete = [path for path in manifests if json.loads(pathlib.Path(path).read_text())['complete']]
assert complete, 'Nenhum treinamento oficial completo encontrado'
manifest_path = pathlib.Path(complete[-1])
manifest = json.loads(manifest_path.read_text())
manifest

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/qwen3_8b_qlora_stage5_evidence', 'zip', manifest_path.parent)
print(archive)
files.download(archive)